# C_2: Transformer Embeddings

Fitur: FinBERT / IndoBERT / LLM (tugas 1a-ii). Data dan pooling sama dengan C_1 supaya bisa dibandingkan langsung.

In [16]:
from c_common import *

D, text = load_all()
print({s: len(d) for s, d in D.items()}, text.shape)

{'train': 825, 'val': 176, 'test': 178} (1202, 9)


In [17]:
#we get finBERT ya
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification
import torch

tok = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert").eval()

def embed(texts, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], padding=True, truncation=True,
                  max_length=512, return_tensors="pt")
        with torch.no_grad():
            h = model(**enc).last_hidden_state[:, 0]   # [CLS] vector, 768 dims
        out.append(h.numpy())
    return np.vstack(out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 256.71it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
texts = ["The central bank raised interest rates, strengthening the dollar.",
         "Oil prices surged after the escalation in the Middle East."]
enc = tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
with torch.no_grad():
    out = model(**enc)

emb = out.last_hidden_state[:, 0]      
print(emb.shape)                       
print(torch.isnan(emb).any())  

torch.Size([2, 768])
tensor(False)


In [19]:
import torch.nn.functional as F

s = ["The dollar strengthened against the rupiah.",
     "The USD rose versus the Indonesian rupiah.",
     "The recipe needs two cups of flour."]
enc = tok(s, padding=True, return_tensors="pt")
with torch.no_grad():
    e = model(**enc).last_hidden_state[:, 0]
print(F.cosine_similarity(e[0:1], e[1:2]))   
print(F.cosine_similarity(e[0:1], e[2:3])) 

tensor([0.6919])
tensor([-0.1528])


## Uji 2 hari nyata dan perbandingan pooling

Sebelum meng-embed semua hari, cek dulu pada 2 hari pertama: bentuk keluaran, ada tidaknya NaN, dan perbedaan pooling `[CLS]` vs rata-rata token.

In [20]:
# Uji pada 2 hari nyata: bentuk, NaN, dan perbandingan pooling CLS vs mean.
# Nama kolom teks tergantung corpus di load_all(): "filtered" -> judul_gabungan, "id" -> judul_gabungan_en.
TITLE = "judul_gabungan" if "judul_gabungan" in text.columns else "judul_gabungan_en"
LEAD = "lead_paragraph_gabungan" if "lead_paragraph_gabungan" in text.columns else "lead_paragraph_gabungan_en"

t2 = text.head(2)
t2_txt = (t2[TITLE].fillna("") + ". " + t2[LEAD].fillna("")).tolist()
print("kolom teks:", TITLE, "+", LEAD)
print("tanggal:", [d.date() for d in t2.index])

enc = tok(t2_txt, padding=True, truncation=True, max_length=512, return_tensors="pt")
print("panjang token:", [len(tok(x)["input_ids"]) for x in t2_txt], "(maks 512, lebih dari itu dipotong)")
with torch.no_grad():
    h = model(**enc).last_hidden_state
m = enc["attention_mask"].unsqueeze(-1)
cls_v = h[:, 0]
mean_v = (h * m).sum(1) / m.sum(1)          # rata-rata token, padding diabaikan

print("bentuk CLS:", tuple(cls_v.shape), "| mean:", tuple(mean_v.shape))
print("ada NaN:", bool(torch.isnan(cls_v).any() or torch.isnan(mean_v).any()))
print("cos(hari1, hari2)  CLS:", round(F.cosine_similarity(cls_v[0:1], cls_v[1:2]).item(), 3),
      "| mean:", round(F.cosine_similarity(mean_v[0:1], mean_v[1:2]).item(), 3))

kolom teks: judul_gabungan_en + lead_paragraph_gabungan_en
tanggal: [datetime.date(2021, 9, 1), datetime.date(2021, 9, 2)]
panjang token: [315, 225] (maks 512, lebih dari itu dipotong)
bentuk CLS: (2, 768) | mean: (2, 768)
ada NaN: False
cos(hari1, hari2)  CLS: 0.554 | mean: 0.88


## Embed semua hari, simpan, lalu evaluasi

Satu vektor per hari dari judul + lead paragraf (konten penuh terlalu panjang untuk batas 512 token). Hari tanpa berita diberi vektor nol dan penanda `has_news`. Dua pooling dicoba (`cls` dan `mean`). Embedding 768 dimensi diperkecil dengan PCA (20 komponen, di-fit hanya di train) supaya tidak overfit pada ~825 hari train.

In [ ]:
# Embed SEMUA hari (jalankan sekali; hasil disimpan di data/features_c/ dan dipakai ulang).
# Di CPU ini bisa memakan waktu beberapa menit. Ubah LIMIT ke angka kecil (mis. 20) untuk uji cepat.
LIMIT = None
FEATURES_C.mkdir(parents=True, exist_ok=True)
f_cls, f_mean = FEATURES_C / "c2_finbert_cls.npy", FEATURES_C / "c2_finbert_mean.npy"

TITLE = "judul_gabungan" if "judul_gabungan" in text.columns else "judul_gabungan_en"
LEAD = "lead_paragraph_gabungan" if "lead_paragraph_gabungan" in text.columns else "lead_paragraph_gabungan_en"
day_text = (text[TITLE].fillna("") + ". " + text[LEAD].fillna("")).str.strip(" .")
has_news = (day_text.str.len() > 0).values

if LIMIT is None and f_cls.exists() and f_mean.exists():
    emb = {"cls": np.load(f_cls), "mean": np.load(f_mean)}
    print("dimuat dari cache:", emb["cls"].shape)
else:
    idx = np.where(has_news)[0][:LIMIT] if LIMIT else np.where(has_news)[0]
    out = {"cls": np.zeros((len(text), 768), dtype="float32"),   # hari tanpa berita = vektor nol
           "mean": np.zeros((len(text), 768), dtype="float32")}
    
    for i in range(0, len(idx), 16):
        b = idx[i:i+16]
        enc = tok(day_text.iloc[b].tolist(), padding=True, truncation=True, max_length=512, return_tensors="pt")
        with torch.no_grad():
            h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        out["cls"][b] = h[:, 0].numpy()
        out["mean"][b] = ((h * m).sum(1) / m.sum(1)).numpy()
        
    emb = out
    if LIMIT is None:                       # jangan simpan cache kalau hanya uji sebagian
        np.save(f_cls, emb["cls"]); np.save(f_mean, emb["mean"])
        print("disimpan ke", FEATURES_C)
print("hari:", len(text), "| dengan berita:", int(has_news.sum()), "| NaN:", bool(np.isnan(emb["cls"]).any()))

disimpan ke ..\data\features_c
hari: 1202 | dengan berita: 1182 | NaN: False


In [23]:
# Gabungkan fitur ke D, lalu nilai di VALIDATION saja (test tidak disentuh).
# PCA di-fit HANYA di data train (hindari kebocoran), lalu diterapkan ke val dan test.
from sklearn.decomposition import PCA

N_COMP = 20
C2_COLS = {}
for pool in ["cls", "mean"]:
    feats = pd.DataFrame(emb[pool], index=text.index)
    train_idx = D["train"]["date"]
    pca = PCA(n_components=N_COMP, random_state=SEED).fit(feats.loc[train_idx].values)
    comp = pd.DataFrame(pca.transform(feats.values), index=text.index,
                        columns=[f"c2_{pool}_{i}" for i in range(N_COMP)])
    comp[f"c2_{pool}_has_news"] = has_news.astype(float)
    for s in D:
        D[s] = D[s].drop(columns=[c for c in comp.columns if c in D[s].columns]) \
                   .merge(comp, left_on="date", right_index=True, how="left")
    C2_COLS[pool] = list(comp.columns)
    print(pool, "variansi dijelaskan PCA:", round(pca.explained_variance_ratio_.sum(), 3))

# n_configs_tried=2 karena kami mencoba 2 pooling (cls, mean) -> dilaporkan jujur di laporan.
for pool in ["cls", "mean"]:
    evaluate_val(f"C2_finbert_{pool}_LR", "LR", D, PRICE + C2_COLS[pool], n_configs_tried=2,
                 note=f"FinBERT {pool} pooling, judul+lead per hari, PCA {N_COMP}")
# Setelah memilih varian terbaik di validation, jalankan final_test(...) SEKALI untuk varian itu saja.

cls variansi dijelaskan PCA: 0.789
mean variansi dijelaskan PCA: 0.717
C2_finbert_cls_LR [LR] val: DA=0.494, BalAcc=0.494, F1_macro=0.494, MCC=-0.012, AUC=0.461
C2_finbert_mean_LR [LR] val: DA=0.483, BalAcc=0.482, F1_macro=0.482, MCC=-0.036, AUC=0.486


In [ ]:
import torch

SEP = "\n\n---\n\n"          # pemisah antar artikel di kolom konten (dibuat di 00d_alignment)
CONTENT = "konten_gabungan" if "konten_gabungan" in text.columns else "konten_gabungan_en"
CHUNK = 510                   # 512 dikurangi token [CLS] dan [SEP]
MAX_CHUNKS = 2                # maks 2 potongan (~1000 token) per artikel; naikkan kalau GPU kuat
BATCH = 16
f_ccls = FEATURES_C / "c2_finbert_content_cls.npy"
f_cmean = FEATURES_C / "c2_finbert_content_mean.npy"


def build_chunks(text_df, tok):
    """Pecah konten tiap hari menjadi artikel, lalu tiap artikel menjadi potongan token.
    Return list of (posisi_hari, id_artikel, token_ids)."""
    rows, art_id = [], 0
    for pos, day in enumerate(text_df[CONTENT].fillna("")):
        for art in [a.strip() for a in day.split(SEP) if a.strip()]:
            ids = tok(art, add_special_tokens=False, truncation=True,
                      max_length=CHUNK * MAX_CHUNKS)["input_ids"]
            for j in range(0, len(ids), CHUNK):
                rows.append((pos, art_id, ids[j:j + CHUNK]))
            art_id += 1
    return rows


@torch.no_grad()
def embed_chunks(rows, tok, model, batch=BATCH):
    """FinBERT untuk semua potongan. Return (CLS, mean) masing-masing (n_potongan, 768)."""
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", dev)
    model.to(dev).eval()
    order = np.argsort([len(r[2]) for r in rows])        # urut menurut panjang -> padding minimal
    cls = np.zeros((len(rows), 768), dtype="float32")
    mean = np.zeros_like(cls)
    for s in range(0, len(order), batch):
        ix = order[s:s + batch]
        enc = tok.pad({"input_ids": [tok.build_inputs_with_special_tokens(rows[i][2]) for i in ix]},
                      return_tensors="pt")
        enc = {k: v.to(dev) for k, v in enc.items()}
        h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        cls[ix] = h[:, 0].float().cpu().numpy()
        mean[ix] = ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()   # padding diabaikan
        if (s // batch) % 200 == 0:
            print(f"  {s}/{len(order)} potongan")
    model.cpu()                                          # supaya cell lama (CPU) tetap jalan
    return cls, mean


def pool_to_days(rows, vecs, n_days):
    """Rata-rata potongan -> artikel, lalu rata-rata artikel -> hari. Hari tanpa artikel = vektor nol."""
    df = pd.DataFrame(vecs)
    df["day"] = [r[0] for r in rows]
    df["art"] = [r[1] for r in rows]
    art_vec = df.groupby(["day", "art"]).mean().reset_index(level="art", drop=True)
    day_vec = art_vec.groupby(level="day").mean()
    out = np.zeros((n_days, vecs.shape[1]), dtype="float32")
    out[day_vec.index.values] = day_vec.values
    return out


FEATURES_C.mkdir(parents=True, exist_ok=True)
if f_ccls.exists() and f_cmean.exists():
    emb_c = {"cls": np.load(f_ccls), "mean": np.load(f_cmean)}
    print("dimuat dari cache:", emb_c["cls"].shape)
else:
    rows = build_chunks(text, tok)
    print(len(rows), "potongan dari", len({r[1] for r in rows}), "artikel")
    c, m = embed_chunks(rows, tok, model)
    emb_c = {"cls": pool_to_days(rows, c, len(text)), "mean": pool_to_days(rows, m, len(text))}
    np.save(f_ccls, emb_c["cls"])
    np.save(f_cmean, emb_c["mean"])
    print("disimpan ke", FEATURES_C)

has_content = (text[CONTENT].fillna("").str.strip().str.len() > 0).values
print("hari:", len(text), "| dengan konten:", int(has_content.sum()), "| NaN:", bool(np.isnan(emb_c["cls"]).any()))

In [ ]:
# Judul+lead PER ARTIKEL: tiap artikel di-embed sendiri, lalu dirata-rata per hari.
# (Versi lama menggabungkan semua judul+lead sehari jadi 1 teks dan dipotong di 512 token; hari dengan
#  banyak artikel kehilangan sebagian besar beritanya.) Judul dan lead dipisah " | " antar artikel.
f_acls = FEATURES_C / "c2_finbert_art_cls.npy"
f_amean = FEATURES_C / "c2_finbert_art_mean.npy"


def build_article_rows(text_df, tok):
    """Satu baris per artikel: "judul. lead" (dipotong 510 token). Return list (posisi_hari, id_artikel, token_ids)."""
    rows, art_id, fallback = [], 0, 0
    for pos, (t, l) in enumerate(zip(text_df[TITLE].fillna(""), text_df[LEAD].fillna(""))):
        titles = [x.strip() for x in t.split(" | ") if x.strip()]
        leads = [x.strip() for x in l.split(" | ")]
        if len(leads) != len(titles):          # jumlah tidak cocok (mis. ada " | " di dalam lead) -> judul saja
            leads, fallback = [""] * len(titles), fallback + 1
        for ti, li in zip(titles, leads):
            ids = tok(f"{ti}. {li}".strip(" ."), add_special_tokens=False, truncation=True, max_length=CHUNK)["input_ids"]
            rows.append((pos, art_id, ids))
            art_id += 1
    print("hari memakai judul saja (jumlah judul/lead tidak cocok):", fallback)
    return rows


if f_acls.exists() and f_amean.exists():
    emb_a = {"cls": np.load(f_acls), "mean": np.load(f_amean)}
    print("dimuat dari cache:", emb_a["cls"].shape)
else:
    rows_a = build_article_rows(text, tok)
    print(len(rows_a), "artikel (judul+lead)")
    c, m = embed_chunks(rows_a, tok, model)
    emb_a = {"cls": pool_to_days(rows_a, c, len(text)), "mean": pool_to_days(rows_a, m, len(text))}
    np.save(f_acls, emb_a["cls"])
    np.save(f_amean, emb_a["mean"])
    print("disimpan ke", FEATURES_C)
print("NaN:", bool(np.isnan(emb_a["cls"]).any()), "| hari dengan vektor:", int((np.abs(emb_a["cls"]).sum(1) > 0).sum()))

In [ ]:
from sklearn.decomposition import PCA

N_COMP = 20


def add_pca_features(emb, prefix, has):
    """PCA di-fit HANYA pada hari train yang punya teks (vektor nol tidak ikut), lalu diterapkan ke semua hari.
    Hari tanpa teks diberi 0 pada semua komponen + penanda has."""
    has = np.asarray(has, bool)
    feats = pd.DataFrame(emb, index=text.index)
    tr = D["train"]["date"]
    tr = tr[has[text.index.get_indexer(tr)]]
    pca = PCA(n_components=N_COMP, random_state=SEED).fit(feats.loc[tr].values)
    z = pca.transform(feats.values)
    z[~has] = 0.0
    comp = pd.DataFrame(z, index=text.index, columns=[f"{prefix}_{i}" for i in range(N_COMP)])
    comp[f"{prefix}_has"] = has.astype(float)
    for s in D:
        D[s] = D[s].drop(columns=[c for c in comp.columns if c in D[s].columns]) \
                   .merge(comp, left_on="date", right_index=True, how="left")
    print(prefix, "variansi dijelaskan PCA:", round(pca.explained_variance_ratio_.sum(), 3))
    return list(comp.columns)


C2C = {p: add_pca_features(emb_c[p], f"c2_content_{p}", has_content) for p in ["cls", "mean"]}
C2A = {p: add_pca_features(emb_a[p], f"c2_art_{p}", has_news) for p in ["cls", "mean"]}

# Sumber teks: "day" = judul+lead digabung 1 teks per hari (versi lama, C2_COLS);
#              "art" = judul+lead per artikel lalu rata-rata harian; "content" = konten per artikel (dipotong) lalu rata-rata harian.
SOURCES = {
    "day":         {p: C2_COLS[p] for p in ["cls", "mean"]},
    "art":         C2A,
    "content":     C2C,
    "art+content": {p: C2A[p] + C2C[p] for p in ["cls", "mean"]},
}
# Nama "day" tetap C2_finbert_{pool}_{kind} supaya cocok dengan file results_c yang sudah ada.
def vname(src, pool, kind):
    return f"C2_finbert_{pool}_{kind}" if src == "day" else f"C2_finbert_{src.replace('+', '_')}_{pool}_{kind}"

KINDS = ["LR", "HGB"]
VARIANTS = {vname(s, p, k): (k, cols[p]) for s, cols in SOURCES.items() for p in ["cls", "mean"] for k in KINDS}
# n_configs_tried = 16: 4 sumber teks x 2 pooling x 2 model. Laporkan jujur.
for name, (kind, cols) in VARIANTS.items():
    evaluate_val(name, kind, D, PRICE + cols, n_configs_tried=len(VARIANTS),
                 note=f"FinBERT, PCA {N_COMP}; " + name)

# Referensi: harga saja (tanpa teks). Representasi teks hanya berguna kalau jelas di atas ini.
for k in KINDS:
    _, sc_price, _ = tune(k, D, PRICE)
    print(f"REFERENSI harga saja ({k}) val:", {a: round(b, 3) for a, b in sc_price.items()})

In [ ]:
import json

recs = {n: json.loads((RESULTS_C / f"{n}.json").read_text()) for n in VARIANTS}
best = max(recs, key=lambda n: recs[n]["val"]["F1_macro"])
kind, cols = VARIANTS[best]
ctrl = shuffled_text_control(kind, D, PRICE, cols, recs[best]["best_params"])
print(best, "| AUC asli (val):", round(recs[best]["val"]["AUC"], 3),
      "| AUC teks diacak:", {k: round(v, 3) for k, v in ctrl.items()})